In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

SCAN_FILE = "out/scan.csv.gz"

df = pd.read_csv(SCAN_FILE, keep_default_na=False)
files = df[df["error"] == ""].copy()
errors = df[df["error"] != ""].copy()

files["size_gb"] = files["size_bytes"] / 1e9
files["mtime"] = pd.to_datetime(files["mtime"])

print(f"Total files:            {len(files):,}")
print(f"Permission denied:      {len(errors):,}")
print(f"Total size:             {files['size_gb'].sum():.1f} GB")

In [ ]:
# Disk usage by owner
by_owner = (
    files.groupby("owner")["size_gb"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(9, 5))
by_owner.plot.bar(ax=ax)
ax.set_xlabel("Owner")
ax.set_ylabel("Total size (GB)")
ax.set_title("Top 20 users by disk usage")
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.xticks(rotation=45, ha="right")
fig.tight_layout()
plt.show()

In [ ]:
# Disk usage by top-level subdirectory
files["top_level"] = files["path"].apply(lambda p: "/" + "/".join(Path(p).parts[1:3]))

by_dir = (
    files.groupby("top_level")["size_gb"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(9, 5))
by_dir.plot.bar(ax=ax)
ax.set_xlabel("Directory")
ax.set_ylabel("Total size (GB)")
ax.set_title("Top 20 directories by disk usage")
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.xticks(rotation=45, ha="right")
fig.tight_layout()
plt.show()

In [ ]:
# File age distribution (by last modification time)
fig, ax = plt.subplots(figsize=(9, 4))
files["mtime"].dt.year.value_counts().sort_index().plot.bar(ax=ax)
ax.set_xlabel("Year last modified")
ax.set_ylabel("Number of files")
ax.set_title("File age distribution")
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
fig.tight_layout()
plt.show()

In [ ]:
# File size distribution (log scale)
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(files["size_bytes"].clip(lower=1), bins=50, log=True)
ax.set_xscale("log")
ax.set_xlabel("File size (bytes)")
ax.set_ylabel("Count")
ax.set_title("File size distribution")
fig.tight_layout()
plt.show()